# Flagship Quant Project: Plastic Pollution & Income

## Research Question
Does a country's income level (GDP per capita) predict how much plastic
pollution it generates per person? Where does Pakistan sit relative to
similar-income countries?

## Data Source
Our World in Data — "Plastic pollution" (Cottom et al., 2024, *Nature*),
2020 snapshot. Covers total plastic pollution (tonnes) by country.

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

df = pd.read_csv('data/plastic-pollution.csv')
df.head()

,Entity,Code,Year,Total plastic pollution
0,Afghanistan,AFG,2020,4.572341e+05
1,Africa (UN M49),NaN,2020,1.559357e+07
2,Akrotiri and Dhekelia,OWID_AKD,2020,5.455690e+00
3,Aland Islands,ALA,2020,6.168265e+00
4,Albania,ALB,2020,2.225899e+04


In [3]:
countries = ['Pakistan', 'India', 'Bangladesh', 'Egypt', 'Nigeria']
df[df['Entity'].isin(countries)]

,Entity,Code,Year,Total plastic pollution
21,Bangladesh,BGD,2020,1748214.60
68,Egypt,EGY,2020,807596.25
105,India,IND,2020,9275777.00
164,Nigeria,NGA,2020,3532479.20
174,Pakistan,PAK,2020,2567460.50


In [4]:
gdp = pd.read_csv('data/gdp-per-capita.csv')
gdp.head()
gdp.columns

Index(['Entity', 'Code', 'Year', 'GDP per capita',
       'World region according to OWID'],
      dtype='str')

In [5]:
# Filter GDP data to just 2020, to match your pollution data's year
gdp_2020 = gdp[gdp['Year'] == 2020]

# Merge the two datasets together, matching rows by country name
merged = pd.merge(df, gdp_2020, on='Entity', how='inner')

merged.head()
print(merged.shape)

(204, 8)


In [6]:
model = smf.ols('Q("Total plastic pollution") ~ Q("GDP per capita")', data=merged).fit()
print(model.summary())

                                 OLS Regression Results                                 
Dep. Variable:     Q("Total plastic pollution")   R-squared:                       0.007
Model:                                      OLS   Adj. R-squared:                  0.002
Method:                           Least Squares   F-statistic:                     1.334
Date:                          Sat, 05 Sep 2026   Prob (F-statistic):              0.249
Time:                                  08:54:28   Log-Likelihood:                -3408.6
No. Observations:                           204   AIC:                             6821.
Df Residuals:                               202   BIC:                             6828.
Df Model:                                     1                                         
Covariance Type:                      nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
----------------------

In [7]:
print(df.columns.tolist())

['Entity', 'Code', 'Year', 'Total plastic pollution']


In [8]:
pop = pd.read_csv('data/population.csv')
pop_2020 = pop[pop['Year'] == 2020]

# Merge population into your existing merged dataset
merged2 = pd.merge(merged, pop_2020[['Entity', 'Population (historical)']], on='Entity', how='inner')

# Create the per-capita pollution column
merged2['pollution_per_capita'] = merged2['Total plastic pollution'] / merged2['Population (historical)']

merged2.head()

KeyError: "['Population (historical)'] not in index"

In [9]:
print(pop.columns.tolist())

['Entity', 'Code', 'Year', 'Population']


In [10]:
pop_2020 = pop[pop['Year'] == 2020]

merged2 = pd.merge(merged, pop_2020[['Entity', 'Population']], on='Entity', how='inner')

merged2['pollution_per_capita'] = merged2['Total plastic pollution'] / merged2['Population']

merged2.head()
print(merged2.shape)

(204, 10)


In [11]:
model2 = smf.ols('pollution_per_capita ~ Q("GDP per capita")', data=merged2).fit()
print(model2.summary())

                             OLS Regression Results                             
Dep. Variable:     pollution_per_capita   R-squared:                       0.505
Model:                              OLS   Adj. R-squared:                  0.502
Method:                   Least Squares   F-statistic:                     205.9
Date:                  Sat, 05 Sep 2026   Prob (F-statistic):           1.17e-32
Time:                          08:58:53   Log-Likelihood:                 821.91
No. Observations:                   204   AIC:                            -1640.
Df Residuals:                       202   BIC:                            -1633.
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept     

Countries with higher income per person tend to generate less plastic pollution per person — this relationship is small in raw magnitude but statistically very strong and reliable

In [12]:
print("R-squared:", model2.rsquared)

R-squared: 0.5047855521880275


## Finding
Across 204 countries, GDP per capita is a strong, statistically significant
predictor of plastic pollution per capita (p < 0.001). Countries with higher
income per person generate substantially less plastic pollution per person,
and income differences alone explain about 50% of the cross-country variation
in per-capita plastic pollution (R² = 0.505).

This supports the idea that pollution is driven less by how much waste a
country produces, and more by whether it has the infrastructure (collection,
management) to handle it — richer countries generally have that infrastructure.